# LSTM + estaticas para binary_paper

Este notebook prueba una alternativa mas simple al Transformer para el caso `binary_paper`:

- secuencia temporal con LSTM
- variables estaticas en una rama densa
- fusion tardia antes de la clasificacion final

Objetivo:

- medir si una arquitectura recurrente + estaticas puede acercarse al Transformer
- comparar sus metricas por semana contra el baseline ya entrenado

In [10]:
from pathlib import Path
import importlib
import sys

import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

In [11]:
PROJECT_ROOT = Path('/workspace/TFM_education_ai_analytics')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from educational_ai_analytics.config import W_WINDOWS
train_transformer = importlib.import_module('educational_ai_analytics.2_modeling.transformers.train_transformer')
load_and_prepare_split = train_transformer.load_and_prepare_split

BASE_NPZ = PROJECT_ROOT / 'data' / '6_transformer_features'
BASELINE_DIR = PROJECT_ROOT / 'reports' / 'transformer_training' / 'binary_classification_paper'
MODEL_DIR = PROJECT_ROOT / 'models' / 'transformers'

NUM_CLASSES = 2
PAPER_BASELINE = True
BINARY_MODE = 'paper'
WITH_STATIC = True
USE_CLUSTERING_FEATURES = False
USE_ACCUMULATED_UPTOW = True
# Cambia a list(W_WINDOWS) si quieres barrer todas las semanas.
WEEKS_TO_RUN = [5]
SEED = 42
EPOCHS = 25
BATCH_SIZE = 64
PATIENCE = 5
LSTM_UNITS = [128, 64]
STATIC_HIDDEN = [64, 32]
HEAD_HIDDEN = [128, 64]
DROPOUT = 0.30
LEARNING_RATE = 1e-3
TUNE_THRESHOLD = False

keras.utils.set_random_seed(SEED)
WEEKS_TO_RUN

[5]

## Carga y ablation de estaticas

Usamos los mismos `.npz` del pipeline Transformer para mantener comparabilidad.

Ademas, podemos apagar columnas de clustering o acumuladas igual que en el entrenamiento del Transformer.

In [12]:
def is_accumulated_name(col: str) -> bool:
    col = str(col)
    accumulated_prefixes = (
        'clicks_', 'weeks_since_', 'streak_', 'total_weighted_', 'last_week_',
        'momentum', 'regularity', 'weekend_share', 'distinct_activity',
        'recency_', 'engagement_', 'pass_ratio', 'late_ratio',
        'submission_', 'avg_score', 'active_weeks',
    )
    return col.startswith(accumulated_prefixes)

def is_cluster_name(col: str) -> bool:
    col = str(col)
    return col.startswith('p_cluster_') or col == 'entropy_norm'

def cluster_drop_mask(feature_names: np.ndarray, feature_sources: np.ndarray, n_features: int, cluster_dim: int) -> np.ndarray:
    if n_features == 0:
        return np.zeros((0,), dtype=bool)
    if feature_sources.size == n_features:
        src = np.array(feature_sources).astype(str)
        return src == 'cluster'
    if feature_names.size == n_features:
        names = np.array(feature_names).astype(str)
        return np.array([is_cluster_name(c) for c in names], dtype=bool)
    fallback = np.zeros((n_features,), dtype=bool)
    if cluster_dim > 0:
        fallback[:min(cluster_dim, n_features)] = True
    return fallback

def filter_static_block(x_static, cluster_dim, feature_names, feature_sources, keep_clusters=True, keep_accumulated=True):
    if x_static is None:
        return None
    n_features = x_static.shape[1]
    keep_mask = np.ones(n_features, dtype=bool)
    drop_cluster = cluster_drop_mask(feature_names, feature_sources, n_features, cluster_dim)
    if not keep_clusters:
        keep_mask &= ~drop_cluster
    if not keep_accumulated:
        if feature_sources.size == n_features:
            keep_mask &= (np.array(feature_sources).astype(str) != 'accumulated_uptow')
        elif feature_names.size == n_features:
            names = np.array(feature_names).astype(str)
            keep_mask &= ~np.array([is_accumulated_name(c) for c in names], dtype=bool)
    if keep_mask.sum() == 0:
        return np.zeros((x_static.shape[0], 0), dtype=x_static.dtype)
    return x_static[:, keep_mask]

def load_split(split: str, week: int):
    X_seq, mask_pad, mask_activity, X_static, y, ids, cluster_dim, static_names, static_sources = load_and_prepare_split(
        BASE_NPZ,
        split,
        week,
        NUM_CLASSES,
        PAPER_BASELINE,
        WITH_STATIC,
        BINARY_MODE,
    )
    if WITH_STATIC and X_static is not None:
        X_static = filter_static_block(
            X_static,
            cluster_dim=cluster_dim,
            feature_names=static_names,
            feature_sources=static_sources,
            keep_clusters=USE_CLUSTERING_FEATURES,
            keep_accumulated=USE_ACCUMULATED_UPTOW,
        )
    return {
        'X_seq': X_seq.astype(np.float32),
        'mask_pad': mask_pad.astype(np.int32),
        'mask_activity': mask_activity.astype(np.int32),
        'X_static': None if X_static is None else X_static.astype(np.float32),
        'y': y.astype(np.int32),
        'ids': ids,
    }

sample = load_split('training', WEEKS_TO_RUN[0])
print('Seq   :', sample['X_seq'].shape)
print('Static:', None if sample['X_static'] is None else sample['X_static'].shape)
print('Y     :', sample['y'].shape)

2026-03-10 17:23:31.801 | INFO     | educational_ai_analytics.2_modeling.transformers.train_transformer:filter_classes:225 - Configuración 2 clases (Baseline Paper): [0: Pass/Dist] vs [1: Withdrawn]. (Fail eliminado)
Seq   : (15099, 5, 20)
Static: (15099, 59)
Y     : (15099,)


## Modelo LSTM + estaticas

La rama temporal usa LSTM apilada y la estatica entra por una MLP corta.
Luego ambas ramas se concatenan antes de la salida binaria.

In [13]:
def build_lstm_static_model(seq_shape, static_dim):
    seq_input = keras.Input(shape=seq_shape, name='seq')
    pad_mask_input = keras.Input(shape=(seq_shape[0],), dtype='bool', name='pad_mask')

    x = keras.layers.LSTM(
        LSTM_UNITS[0],
        return_sequences=len(LSTM_UNITS) > 1,
        dropout=0.0,
        recurrent_dropout=0.0,
        name='lstm_1',
    )(seq_input, mask=pad_mask_input)

    for index, units in enumerate(LSTM_UNITS[1:], start=2):
        return_sequences = index < len(LSTM_UNITS)
        x = keras.layers.LSTM(
            units,
            return_sequences=return_sequences,
            dropout=0.0,
            recurrent_dropout=0.0,
            name=f'lstm_{index}',
        )(x)

    seq_repr = keras.layers.Dense(64, activation='relu', name='seq_repr')(x)
    seq_repr = keras.layers.Dropout(DROPOUT, name='seq_dropout')(seq_repr)

    inputs = [seq_input, pad_mask_input]
    if static_dim and static_dim > 0:
        static_input = keras.Input(shape=(static_dim,), name='static')
        s = static_input
        for index, units in enumerate(STATIC_HIDDEN):
            s = keras.layers.Dense(units, activation='relu', name=f'static_dense_{index + 1}')(s)
            s = keras.layers.Dropout(DROPOUT, name=f'static_dropout_{index + 1}')(s)
        fused = keras.layers.Concatenate(name='fusion')([seq_repr, s])
        inputs.append(static_input)
    else:
        fused = seq_repr

    h = fused
    for index, units in enumerate(HEAD_HIDDEN):
        h = keras.layers.Dense(units, activation='relu', name=f'head_dense_{index + 1}')(h)
        h = keras.layers.Dropout(DROPOUT, name=f'head_dropout_{index + 1}')(h)

    output = keras.layers.Dense(1, activation='sigmoid', name='risk_prob')(h)

    model = keras.Model(inputs=inputs, outputs=output, name='lstm_static_binary_paper')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=[
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
        ],
    )
    return model

preview_model = build_lstm_static_model(
    seq_shape=sample['X_seq'].shape[1:],
    static_dim=0 if sample['X_static'] is None else sample['X_static'].shape[1],
)
preview_model.summary()

Model: "lstm_static_binary_paper"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ seq (InputLayer)    │ (None, 5, 20)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pad_mask            │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ static (InputLayer) │ (None, 59)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 5, 128)    │     76,288 │ seq[0][0],        │
│                     │                   │            │ pad_mask[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ static_dense_1      │ (None, 64)        │      3,840 │ static[0][0]      │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 64)        │     49,408 │ lstm_1[0][0],     │
│                     │                   │            │ pad_mask[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ static_dropout_1    │ (None, 64)        │          0 │ static_dense_1[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ seq_repr (Dense)    │ (None, 64)        │      4,160 │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ static_dense_2      │ (None, 32)        │      2,080 │ static_dropout_1… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ seq_dropout         │ (None, 64)        │          0 │ seq_repr[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ static_dropout_2    │ (None, 32)        │          0 │ static_dense_2[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion              │ (None, 96)        │          0 │ seq_dropout[0][0… │
│ (Concatenate)       │                   │            │ static_dropout_2… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ head_dense_1        │ (None, 128)       │     12,416 │ fusion[0][0]      │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ head_dropout_1      │ (None, 128)       │          0 │ head_dense_1[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ head_dense_2        │ (None, 64)        │      8,256 │ head_dropout_1[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ head_dropout_2      │ (None, 64)        │          0 │ head_dense_2[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ risk_prob (Dense)   │ (None, 1)         │         65 │ head_dropout_2[0… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 156,513 (611.38 KB)

 Trainable params: 156,513 (611.38 KB)

 Non-trainable params: 0 (0.00 B)

## Entrenamiento por semana

Por defecto se entrena una corrida por semana de `WEEKS_TO_RUN`, con early stopping sobre `val_auc`.
Si quieres una prueba rapida, deja `WEEKS_TO_RUN = [5]` arriba.

In [14]:
def build_inputs(split_payload):
    inputs = [split_payload['X_seq'], split_payload['mask_pad'].astype(bool)]
    if split_payload['X_static'] is not None and split_payload['X_static'].shape[1] > 0:
        inputs.append(split_payload['X_static'])
    return inputs

def find_best_threshold(y_true, y_prob, thresholds=None):
    thresholds = np.linspace(0.1, 0.9, 161) if thresholds is None else thresholds
    best_threshold = 0.5
    best_bacc = -1.0
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        score = balanced_accuracy_score(y_true, y_pred)
        if score > best_bacc:
            best_bacc = float(score)
            best_threshold = float(threshold)
    return best_threshold, best_bacc

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1_score': float(f1_score(y_true, y_pred, zero_division=0)),
        'auc': float(roc_auc_score(y_true, y_prob)),
    }

def train_week(week: int):
    train_payload = load_split('training', week)
    val_payload = load_split('validation', week)
    test_payload = load_split('test', week)

    static_dim = 0 if train_payload['X_static'] is None else train_payload['X_static'].shape[1]
    model = build_lstm_static_model(train_payload['X_seq'].shape[1:], static_dim)

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_auc',
            mode='max',
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1,
        )
    ]

    class_counts = np.bincount(train_payload['y'])
    class_weight = {
        int(index): float(len(train_payload['y']) / (len(class_counts) * count))
        for index, count in enumerate(class_counts) if count > 0
    }

    history = model.fit(
        x=build_inputs(train_payload),
        y=train_payload['y'],
        validation_data=(build_inputs(val_payload), val_payload['y']),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=0,
    )

    val_prob = model.predict(build_inputs(val_payload), verbose=0).reshape(-1)
    selected_threshold = 0.5
    if TUNE_THRESHOLD:
        selected_threshold, _ = find_best_threshold(val_payload['y'], val_prob)

    test_prob = model.predict(build_inputs(test_payload), verbose=0).reshape(-1)
    val_metrics = compute_metrics(val_payload['y'], val_prob, threshold=selected_threshold)
    test_metrics = compute_metrics(test_payload['y'], test_prob, threshold=selected_threshold)

    return {
        'week': int(week),
        'epochs_trained': int(len(history.history['loss'])),
        'selected_threshold': float(selected_threshold),
        **{f'val_{k}': v for k, v in val_metrics.items()},
        **{f'test_{k}': v for k, v in test_metrics.items()},
    }

results = [train_week(week) for week in WEEKS_TO_RUN]
results_df = pd.DataFrame(results).sort_values('week').reset_index(drop=True)
results_df

2026-03-10 17:23:32.048 | INFO     | educational_ai_analytics.2_modeling.transformers.train_transformer:filter_classes:225 - Configuración 2 clases (Baseline Paper): [0: Pass/Dist] vs [1: Withdrawn]. (Fail eliminado)
2026-03-10 17:23:32.077 | INFO     | educational_ai_analytics.2_modeling.transformers.train_transformer:filter_classes:225 - Configuración 2 clases (Baseline Paper): [0: Pass/Dist] vs [1: Withdrawn]. (Fail eliminado)
2026-03-10 17:23:32.103 | INFO     | educational_ai_analytics.2_modeling.transformers.train_transformer:filter_classes:225 - Configuración 2 clases (Baseline Paper): [0: Pass/Dist] vs [1: Withdrawn]. (Fail eliminado)
Epoch 11: early stopping
Restoring model weights from the end of the best epoch: 6.


,week,epochs_trained,selected_threshold,val_accuracy,val_balanced_accuracy,val_precision,val_recall,val_f1_score,val_auc,test_accuracy,test_balanced_accuracy,test_precision,test_recall,test_f1_score,test_auc
0,5,11,0.5,0.809656,0.759436,0.70876,0.631846,0.668097,0.826959,0.808414,0.75495,0.694064,0.624872,0.657653,0.831721


## Comparativa contra el Transformer base

Comparamos las metricas de este LSTM en test con el resumen ya exportado del Transformer `binary_paper`.

In [15]:
baseline_test_df = pd.read_csv(BASELINE_DIR / 'test_summary_2clases_paper.csv')
comparison_df = results_df.rename(columns={'week': 'upto_week'}).merge(
    baseline_test_df,
    on='upto_week',
    suffixes=('_lstm', '_transformer'),
    how='left',
)

metrics_to_compare = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'auc']
for metric in metrics_to_compare:
    comparison_df[f'delta_{metric}'] = comparison_df[f'test_{metric}'] - comparison_df[f'{metric}']

comparison_view = comparison_df[[
    'upto_week',
    'test_accuracy', 'accuracy', 'delta_accuracy',
    'test_balanced_accuracy', 'balanced_accuracy', 'delta_balanced_accuracy',
    'test_precision', 'precision', 'delta_precision',
    'test_recall', 'recall', 'delta_recall',
    'test_auc', 'auc', 'delta_auc',
]].copy()
comparison_view = comparison_view.sort_values('upto_week').reset_index(drop=True)
comparison_view

,upto_week,test_accuracy,accuracy,delta_accuracy,test_balanced_accuracy,balanced_accuracy,delta_balanced_accuracy,test_precision,precision,delta_precision,test_recall,recall,delta_recall,test_auc,auc,delta_auc
0,5,0.808414,0.791465,0.016949,0.75495,0.754015,0.000935,0.694064,0.641153,0.052911,0.624872,0.662898,-0.038027,0.831721,0.826103,0.005618


In [16]:
summary_rows = []
for metric in metrics_to_compare:
    delta_col = f'delta_{metric}'
    summary_rows.append({
        'metric': metric,
        'mean_delta': float(comparison_df[delta_col].mean()),
        'median_delta': float(comparison_df[delta_col].median()),
        'max_gain': float(comparison_df[delta_col].max()),
        'max_loss': float(comparison_df[delta_col].min()),
        'weeks_won': int((comparison_df[delta_col] > 0).sum()),
        'weeks_lost': int((comparison_df[delta_col] < 0).sum()),
    })

contrast_summary_df = pd.DataFrame(summary_rows).sort_values('mean_delta', ascending=False)
contrast_summary_df

,metric,mean_delta,median_delta,max_gain,max_loss,weeks_won,weeks_lost
2,precision,0.052911,0.052911,0.052911,0.052911,1,0
0,accuracy,0.016949,0.016949,0.016949,0.016949,1,0
4,auc,0.005618,0.005618,0.005618,0.005618,1,0
1,balanced_accuracy,0.000935,0.000935,0.000935,0.000935,1,0
3,recall,-0.038027,-0.038027,-0.038027,-0.038027,0,1


## Lectura rapida

Si el LSTM se acerca al Transformer o incluso gana en algunas semanas, habra senal de que una arquitectura recurrente todavia es competitiva en este problema.

Si queda claramente por debajo, el Transformer seguira siendo la mejor referencia y el LSTM nos servira como baseline mas simple.

In [17]:
print('LSTM results:')
display(results_df)
print('\nComparison vs transformer:')
display(comparison_view)
print('\nContrast summary:')
display(contrast_summary_df)

LSTM results:


,week,epochs_trained,selected_threshold,val_accuracy,val_balanced_accuracy,val_precision,val_recall,val_f1_score,val_auc,test_accuracy,test_balanced_accuracy,test_precision,test_recall,test_f1_score,test_auc
0,5,11,0.5,0.809656,0.759436,0.70876,0.631846,0.668097,0.826959,0.808414,0.75495,0.694064,0.624872,0.657653,0.831721



Comparison vs transformer:


,upto_week,test_accuracy,accuracy,delta_accuracy,test_balanced_accuracy,balanced_accuracy,delta_balanced_accuracy,test_precision,precision,delta_precision,test_recall,recall,delta_recall,test_auc,auc,delta_auc
0,5,0.808414,0.791465,0.016949,0.75495,0.754015,0.000935,0.694064,0.641153,0.052911,0.624872,0.662898,-0.038027,0.831721,0.826103,0.005618



Contrast summary:


,metric,mean_delta,median_delta,max_gain,max_loss,weeks_won,weeks_lost
2,precision,0.052911,0.052911,0.052911,0.052911,1,0
0,accuracy,0.016949,0.016949,0.016949,0.016949,1,0
4,auc,0.005618,0.005618,0.005618,0.005618,1,0
1,balanced_accuracy,0.000935,0.000935,0.000935,0.000935,1,0
3,recall,-0.038027,-0.038027,-0.038027,-0.038027,0,1
